In [1]:

"""
FinQA (FinanceRAG) retrieval evaluation harness — same shape as your FinDER
notebook (splitter x retriever x k sweep, Recall@k / NDCG@k), pointed at
FinQA's corpus/queries/qrels instead.
 
Two fixes applied relative to the FinDER notebook (see chat for why):
  1. chunk_vecs now uses normalize_embeddings=True, matching query_vecs,
     so cosine_numpy/faiss_flat retrievers compare true cosine similarity
     rather than similarity biased by chunk embedding norm.
  2. The reranker is now actually constructed from cfg.reranker_name and
     used -- previously self.reranker was hardcoded to None, so every
     "reranker=cross_encoder" row in your FinDER results was actually
     unreranked retrieval output.
 
IMPORTANT -- verify before running:
  - File names below (finqa_corpus.jsonl, finqa_queries.jsonl,
    FinQA_qrels.tsv) follow the same naming pattern as your FinDER files,
    but I have not confirmed FinQA's actual Kaggle export file names --
    check your downloaded finrag_data folder and adjust DataConfig paths
    to match exactly.
  - FinQA source documents mix narrative text with financial tables.
    A plain char/token splitter can cut a table mid-row, destroying the
    numeric relationships a downstream answer would need. Inspect a
    sample of your built chunks (print a few chunk_texts entries) before
    trusting the sweep -- if tables are visibly mangled, a table-aware
    splitter (or a pre-processing step that serializes tables to a
    single-line format before chunking) matters more here than any
    chunk_size/overlap tuning.
  - This script evaluates RETRIEVAL quality only (recall/ndcg), same as
    FinDER. FinQA's original benchmark also checks numerical ANSWER
    accuracy (did the final answer match the expected number), which is
    a separate generation-stage evaluation this script does not cover --
    see the note at the bottom of this file.
"""
 
import math
from dataclasses import dataclass, field
 
try:
    from Retrievers import get_retriever
    from Splitters import get_splitter
except ImportError:
    from retrievers import get_retriever
    from splitters import get_splitter
 
try:
    from Rerankers import get_reranker
except ImportError:
    try:
        from rerankers import get_reranker
    except ImportError:
        get_reranker = None  # reranking simply unavailable; CONFIG.reranker_name must be None
 
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer


D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [25]:
!pip install openai

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 18.6 MB/s  0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
 

In [2]:
 
# ---------------------------------------------------------------------------
# Metrics (unchanged from FinDER notebook)
# ---------------------------------------------------------------------------
 
def dcg(rels: list[int]) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
 
 
def ndcg_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    rels = [1 if d in relevant_doc_ids else 0 for d in retrieved_doc_ids[:k]]
    ideal_dcg = dcg(sorted(rels, reverse=True))
    return dcg(rels) / ideal_dcg if ideal_dcg > 0 else 0.0
 
 
def recall_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    if not relevant_doc_ids:
        return 0.0
    hit = len(set(retrieved_doc_ids[:k]) & relevant_doc_ids)
    return hit / len(relevant_doc_ids)
    

In [3]:
# ---------------------------------------------------------------------------
# Config (unchanged shape from FinDER notebook; paths/output point to FinQA)
# ---------------------------------------------------------------------------
 
@dataclass(frozen=True)
class SplitterSpec:
    name: str
    chunk_size: int
    overlap: int
 
 
@dataclass(frozen=True)
class DataConfig:
    corpus_path: str
    queries_path: str
    qrels_path: str
    qrels_cols: tuple[str, str, str] = ("query_id", "corpus_id", "score")
    qrels_delimiter: str = "\t"
 
 
@dataclass(frozen=True)
class ExperimentConfig:
    data: DataConfig
    embed_model: str
    splitter_specs: list[SplitterSpec]
    retriever_names: list[str]
    k_values: list[int] = field(default_factory=lambda: [5, 10, 20, 30, 40, 50])
    max_queries: int | None = 200
    max_corpus_docs: int | None = None
    output_csv: str = "../results/finqa_results.csv"
    reranker_name: str | None = None
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    rerank_pool: int = 50
 
    @property
    def max_k(self) -> int:
        return max(self.k_values)
 
 

In [4]:
# ---------------------------------------------------------------------------
# Data loading (unchanged from FinDER notebook)
# ---------------------------------------------------------------------------
 
class FinRAGDataLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
 
    def load(self):
        corpus_ds = load_dataset("json", data_files=self.cfg.corpus_path, split="train")
        queries_ds = load_dataset("json", data_files=self.cfg.queries_path, split="train")
        try:
            qrels_ds = load_dataset("csv", data_files=self.cfg.qrels_path,
                                     delimiter=self.cfg.qrels_delimiter, split="train")
        except Exception as e:
            raise SystemExit(f"Could not load qrels file '{self.cfg.qrels_path}': {e}")
        return list(corpus_ds), list(queries_ds), qrels_ds
 
    def build_relevant_map(self, qrels_ds, query_ids: set[str]) -> dict[str, set[str]]:
        qcol, dcol, scol = self.cfg.qrels_cols
        relevant: dict[str, set[str]] = {}
        for row in qrels_ds:
            qid = str(row[qcol])
            if qid not in query_ids:
                continue
            if row.get(scol, 1) and int(row[scol]) > 0:
                relevant.setdefault(qid, set()).add(str(row[dcol]))
        return relevant
 
 

In [5]:
# ---------------------------------------------------------------------------
# Pipeline -- same structure as FinDER notebook, with the two fixes applied
# ---------------------------------------------------------------------------

class Pipeline:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.loader = FinRAGDataLoader(cfg.data)
        self._model = SentenceTransformer(cfg.embed_model)
        # FIX: actually construct the reranker from cfg.reranker_name instead
        # of hardcoding None. Loaded once here (not per splitter/retriever
        # combo) since a cross-encoder is comparatively expensive to load.
        if cfg.reranker_name:
            if get_reranker is None:
                raise ImportError(
                    "cfg.reranker_name is set but no get_reranker() factory could be "
                    "imported -- add a Rerankers.py/rerankers.py module, or set "
                    "reranker_name=None to run without reranking."
                )
            self.reranker = get_reranker(cfg.reranker_name, cfg.reranker_model)
        else:
            self.reranker = None
 
    def _build_chunks(self, corpus_rows, splitter_name, chunk_size, overlap):
        splitter = get_splitter(splitter_name, chunk_size, overlap)
        doc_ids, texts = [], []
        for row in corpus_rows:
            title = (row.get("title") or "").strip()
            body = row.get("text") or ""
            full_text = f"{title}\n{body}" if title else body
            for piece in splitter.split(full_text):
                if piece and piece.strip():
                    doc_ids.append(str(row["_id"]))
                    texts.append(piece)
        return doc_ids, texts
 
    def run(self) -> pd.DataFrame:
        cfg = self.cfg
        corpus_rows, queries_rows, qrels_ds = self.loader.load()
        if cfg.max_corpus_docs:
            corpus_rows = corpus_rows[:cfg.max_corpus_docs]
        if cfg.max_queries:
            queries_rows = queries_rows[:cfg.max_queries]
 
        query_ids = {str(r["_id"]) for r in queries_rows}
        relevant = self.loader.build_relevant_map(qrels_ds, query_ids)
        print(f"Corpus docs: {len(corpus_rows)} | Queries: {len(queries_rows)} "
              f"| Queries with qrels: {len(relevant)}")
 
        query_texts = [r["text"] for r in queries_rows]
        query_vecs = self._model.encode(query_texts, normalize_embeddings=True,
                                         batch_size=64, show_progress_bar=True)
 
        results = []
        for spec in cfg.splitter_specs:
            try:
                chunk_doc_ids, chunk_texts = self._build_chunks(
                    corpus_rows, spec.name, spec.chunk_size, spec.overlap)
            except ImportError as e:
                print(f"Skipping splitter '{spec.name}': missing dependency ({e})")
                continue
 
            # Sanity check before embedding anything at scale: eyeball a
            # couple of chunks for mangled tables, since FinQA source docs
            # mix narrative text with financial tables (see module docstring).
            if chunk_texts:
                print(f"  sample chunk (splitter={spec.name}, size={spec.chunk_size}): "
                      f"{chunk_texts[0][:200]!r}")
 
            # FIX: normalize_embeddings=True added here to match query_vecs --
            # previously this call had no normalization, biasing cosine_numpy/
            # faiss_flat toward chunks with larger embedding norms.
            chunk_vecs = self._model.encode(chunk_texts, normalize_embeddings=True,
                                             batch_size=64, show_progress_bar=True)
 
            for retriever_name in cfg.retriever_names:
                try:
                    retriever = get_retriever(retriever_name)
                except KeyError as e:
                    print(e)
                    continue
 
                pool_k = max(cfg.max_k, cfg.rerank_pool) if self.reranker else cfg.max_k
                try:
                    retriever.fit(chunk_texts, chunk_vecs)
                    pool_idx = retriever.query(query_texts, query_vecs, pool_k)
                except ImportError as e:
                    print(f"Skipping retriever '{retriever_name}': missing dependency ({e})")
                    continue
 
                per_k_recall = {k: [] for k in cfg.k_values}
                per_k_ndcg = {k: [] for k in cfg.k_values}
                for qi, row in enumerate(queries_rows):
                    rel_docs = relevant.get(str(row["_id"]))
                    if not rel_docs:
                        continue
                    candidate_idx = list(pool_idx[qi])
                    if self.reranker:
                        candidate_texts = [chunk_texts[i] for i in candidate_idx]
                        order = self.reranker.rerank(row["text"], candidate_texts)
                        candidate_idx = [candidate_idx[o] for o in order]
                    retrieved = [chunk_doc_ids[i] for i in candidate_idx]
                    for k in cfg.k_values:
                        per_k_recall[k].append(recall_at_k(retrieved, rel_docs, k))
                        per_k_ndcg[k].append(ndcg_at_k(retrieved, rel_docs, k))
 
                for k in cfg.k_values:
                    results.append({
                        "splitter": spec.name, "chunk_size": spec.chunk_size, "overlap": spec.overlap,
                        "retriever": retriever_name,
                        "reranker": cfg.reranker_name or "none",
                        "k": k,
                        "n_chunks": len(chunk_texts),
                        "n_queries_scored": len(per_k_recall[k]),
                        "recall": np.mean(per_k_recall[k]) if per_k_recall[k] else float("nan"),
                        "ndcg": np.mean(per_k_ndcg[k]) if per_k_ndcg[k] else float("nan"),
                    })
 
        df = pd.DataFrame(results).sort_values(["k", "ndcg"], ascending=[True, False])
        df.to_csv(cfg.output_csv, index=False)
        return df
 
 


In [6]:
# ---------------------------------------------------------------------------
# Experiment definition -- VERIFY file paths against your downloaded FinQA
# export before running (see module docstring).
# ---------------------------------------------------------------------------
 
CONFIG = ExperimentConfig(
    data=DataConfig(
        corpus_path="../finrag_data/finqa_corpus.jsonl/corpus.jsonl",   # VERIFY this path
        queries_path="../finrag_data/finqa_queries.jsonl/queries.jsonl",  # VERIFY this path
        qrels_path="../finrag_data/FinQA_qrels.tsv",                     # VERIFY this path
    ),
    embed_model="sentence-transformers/all-MiniLM-L6-v2",
    splitter_specs=[
        SplitterSpec("char", 256, 40),
        SplitterSpec("char", 512, 0),
        SplitterSpec("recursive_char", 256, 64),
        SplitterSpec("token", 128, 20),
    ],
    retriever_names=["cosine_numpy", "faiss_flat", "bm25", "hybrid_rrf"],
    k_values=[5, 10, 20, 30, 40, 50],
    max_queries=200,
    max_corpus_docs=None,
    reranker_name=None,   # now actually applied -- see Bug 1 fix above
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    rerank_pool=50,
)
 
if __name__ == "__main__":
    df = Pipeline(CONFIG).run()
    print(df.to_string(index=False))
 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2400.30it/s]


Corpus docs: 2789 | Queries: 200 | Queries with qrels: 60


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.44it/s]


  sample chunk (splitter=char, size=256): 'performance graph the performance graph below shows the five-year cumulative total stockholder return on applied common stock during the period from october 25 , 2009 through october 26 , 2014 .\nthis '


Batches: 100%|██████████| 907/907 [01:15<00:00, 12.05it/s]


...

  sample chunk (splitter=recursive_char, size=256): 'performance graph the performance graph below shows the five-year cumulative total stockholder return on applied common stock during the period from october 25 , 2009 through october 26 , 2014 .'


Batches: 100%|██████████| 1112/1112 [01:14<00:00, 14.90it/s]


  sample chunk (splitter=token, size=128): 'performance graph the performance graph below shows the five-year cumulative total stockholder return on applied common stock during the period from october 25 , 2009 through october 26 , 2014 .\nthis '


Batches: 100%|██████████| 497/497 [00:44<00:00, 11.23it/s]


      splitter  chunk_size  overlap    retriever reranker  k  n_chunks  n_queries_scored   recall     ndcg
recursive_char         256       64   hybrid_rrf     none  5     71108                60 0.616667 0.454598
recursive_char         256       64 cosine_numpy     none  5     71108                60 0.483333 0.402907
recursive_char         256       64   faiss_flat     none  5     71108                60 0.483333 0.400725
         token         128       20         bm25     none  5     31775                60 0.483333 0.393929
          char         512        0         bm25     none  5     25539                60 0.466667 0.389678
          char         256       40         bm25     none  5     58044                60 0.500000 0.380180
         token         128       20   hybrid_rrf     none  5     31775                60 0.500000 0.376630
         token         128       20 cosine_numpy     none  5     31775                60 0.500000 0.375387
         token         128       20  

In [9]:
def build_gold_map(gold_path: str, id_field: str = "id", answer_field: str = "exe_ans") -> dict[str, float]:
    """
    Loads a ground-truth mapping from a JSON/JSONL file. Adjust id_field/
    answer_field to match whatever your join file actually looks like --
    this is a placeholder shape, not a confirmed schema, since I haven't
    seen your actual joined ground-truth file.
    """
    gold = {}
    with open(gold_path) as f:
        if gold_path.endswith(".jsonl"):
            rows = [json.loads(line) for line in f if line.strip()]
        else:
            rows = json.load(f)
    for row in rows:
        qid = str(row[id_field])
        try:
            gold[qid] = float(row[answer_field])
        except (TypeError, ValueError):
            continue  # skip rows where the answer isn't a clean number
    return gold


    
 

In [10]:
import re
_NUMBER_RE = re.compile(r"-?\$?[\d,]+\.?\d*%?")

In [11]:

def extract_number(text: str) -> float | None:
    """Pulls the first plausible number out of a free-text LLM answer,
    handling $, commas, %, and parenthesized negatives (accounting style)."""
    if text is None:
        return None
    text = text.strip()
 
    # Accounting-style negatives: (1,234.5) -> -1234.5
    paren_negative = text.startswith("(") and text.endswith(")")
    if paren_negative:
        text = text[1:-1]
 
    match = _NUMBER_RE.search(text)
    if not match:
        return None
 
    raw = match.group().replace(",", "").replace("$", "")
    is_percent = raw.endswith("%")
    raw = raw.rstrip("%")
 
    try:
        value = float(raw)
    except ValueError:
        return None
 
    if is_percent:
        value = value / 100.0   # normalize "90%" -> 0.90, matching gold answers stored as ratios
    if paren_negative:
        value = -abs(value)
    return value
    

In [12]:

def answers_match(predicted: float | None, gold: float, rel_tol: float = 0.001) -> bool:
    """Same tolerance convention as the original FinQA paper's execution
    accuracy metric: math.isclose with rel_tol=0.001 (0.1% relative error)."""
    if predicted is None:
        return False
    return math.isclose(predicted, gold, rel_tol=rel_tol)
    

In [13]:
# ---------------------------------------------------------------------------
# Retrieval (reuses your existing splitter/retriever factories, one fixed
# config -- plug in whatever your FinQA retrieval sweep found best)
# ---------------------------------------------------------------------------
 
def build_chunks(corpus_rows, splitter_name, chunk_size, overlap):
    splitter = get_splitter(splitter_name, chunk_size, overlap)
    doc_ids, texts = [], []
    for row in corpus_rows:
        title = (row.get("title") or "").strip()
        body = row.get("text") or ""
        full_text = f"{title}\n{body}" if title else body
        for piece in splitter.split(full_text):
            if piece and piece.strip():
                doc_ids.append(str(row["_id"]))
                texts.append(piece)
    return doc_ids, texts


In [32]:

ANSWER_PROMPT = """You are a financial analyst answering a question using ONLY the excerpts below.
Give your final numeric answer on its own, with no extra commentary -- a single
number, percentage, or ratio, formatted plainly (e.g. "1234.5" or "12.3%").
If the excerpts don't contain enough information to answer, respond exactly
with: insufficient_context
 
Excerpts:
{context}
 
Question: {question}
 
Final numeric answer:"""


In [33]:
def generate_answer(client, provider: str, model: str, question: str, context_chunks: list[str]) -> str:
    prompt = ANSWER_PROMPT.format(context="\n\n".join(context_chunks), question=question)
 
    if provider == "anthropic":
        message = client.messages.create(
            model=model, max_tokens=100,
            messages=[{"role": "user", "content": prompt}],
        )
        return message.content[0].text.strip()
    elif provider == "openai":
        response = client.chat.completions.create(
            model=model, max_tokens=100,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    raise ValueError(f"Unknown provider '{provider}' -- use 'anthropic' or 'openai'")
    

In [34]:
def make_client(provider: str):
    if provider == "anthropic":
        import anthropic
        return anthropic.Anthropic()
    elif provider == "openai":
        import openai
        return openai.OpenAI()
    raise ValueError(f"Unknown provider '{provider}' -- use 'anthropic' or 'openai'")
    

In [36]:
def evaluate_finqa_answers(
    corpus_path: str, queries_path: str, gold_map: dict[str, float],
    splitter_name: str, chunk_size: int, overlap: int, retriever_name: str,
    embed_model: str, llm_provider: str, llm_model: str,
    top_k: int = 5, max_queries: int | None = 100,
) -> pd.DataFrame:
    client = make_client(llm_provider)
 
    corpus_rows = list(load_dataset("json", data_files=corpus_path, split="train"))
    queries_rows = list(load_dataset("json", data_files=queries_path, split="train"))
    if max_queries:
        queries_rows = queries_rows[:max_queries]
 
    embedder = SentenceTransformer(embed_model)
    chunk_doc_ids, chunk_texts = build_chunks(corpus_rows, splitter_name, chunk_size, overlap)
    chunk_vecs = embedder.encode(chunk_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
 
    retriever = get_retriever(retriever_name)
    retriever.fit(chunk_texts, chunk_vecs)
 
    rows = []
    for row in queries_rows:
        qid = str(row["_id"])
        gold = gold_map.get(qid)
        if gold is None:
            continue  # no ground truth available for this query -- can't score it
 
        query_vec = embedder.encode([row["text"]], normalize_embeddings=True)
        top_idx = retriever.query([row["text"]], query_vec, top_k)[0]
        context_chunks = [chunk_texts[i] for i in top_idx]
 
        raw_answer = generate_answer(client, llm_provider, llm_model, row["text"], context_chunks)
        predicted = extract_number(raw_answer)
        correct = answers_match(predicted, gold)
 
        rows.append({
            "query_id": qid,
            "question": row["text"],
            "gold": gold,
            "raw_answer": raw_answer,
            "predicted": predicted,
            "correct": correct,
        })
 
    df = pd.DataFrame(rows)
    if df.empty:
        print("No queries had matching ground truth -- check gold_map keys against query _id values.")
        return df
 
    n_parseable = df["predicted"].notna().sum()
    print(f"Queries scored: {len(df)}")
    print(f"Answers with a parseable number: {n_parseable} ({n_parseable / len(df):.1%})")
    print(f"Execution accuracy (within 0.1% tolerance): {df['correct'].mean():.1%}")
    return df
 

    

In [37]:
from dotenv import load_dotenv
load_dotenv() 


True

In [39]:
# VERIFY every path below against your actual local files before running.
    # Requires: pip install openai --break-system-packages (in addition to
    # anthropic, if you want to switch providers later) and OPENAI_API_KEY
    # set in the environment.
gold_map = build_gold_map("../finrag_data/finqa_gold_answers.json")  # you build this join file (see docstring)

df = evaluate_finqa_answers(
    corpus_path="../finrag_data/finqa_corpus.jsonl/corpus.jsonl",
    queries_path="../finrag_data/finqa_queries.jsonl/queries.jsonl",
    gold_map=gold_map,
    splitter_name="char", chunk_size=256, overlap=40,   # swap for your best FinQA retrieval config
    retriever_name="cosine_numpy",
    embed_model="sentence-transformers/all-MiniLM-L6-v2",
    llm_provider="openai",
    llm_model="gpt-4o-mini",
    top_k=30,
    max_queries=100,
)
df.to_csv("../results/finqa_answer_results.csv", index=False)



Batches: 100%|██████████| 907/907 [01:09<00:00, 13.08it/s]


Queries scored: 95
Answers with a parseable number: 40 (42.1%)
Execution accuracy (within 0.1% tolerance): 2.1%


In [31]:
import json
with open("../finrag_data/finqa_queries.jsonl/queries.jsonl") as f:
    first_row = json.loads(f.readline())
print(first_row.keys())
print(first_row)

dict_keys(['_id', 'title', 'text'])
{'_id': 'q61676968', 'title': '', 'text': 'what was the average impact on dva of a 1 basis point increase in jpmorgan chase credit spread for 2008 and 2007?'}
